In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, AllChem
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.manifold import TSNE
from tqdm.auto import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

In [ ]:
import platform
if platform.system() == 'Darwin': # Mac
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows': # Windows
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

In [ ]:
print("===== 1. 데이터 로드 시작 =====")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

try:
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('test.csv')
    submission_df = pd.read_csv('sample_submission.csv')
except FileNotFoundError:
    print("경고: csv 파일을 찾을 수 없습니다. 임시 예제 데이터로 분석을 진행합니다.")
    train_df = pd.DataFrame({'ID': [f'train_{i}' for i in range(100)], 'Canonical_SMILES': ['CCO', 'CCC', 'CCN', 'CC(=O)O', 'c1ccccc1'] * 20, 'Inhibition': np.random.rand(100) * 100})
    test_df = pd.DataFrame({'ID': [f'test_{i}' for i in range(50)], 'Canonical_SMILES': ['CNC', 'CCOc1ccccc1', 'CC(C)C', 'C1CCCCC1'] * 12 + ['C1=CC=C(C=C1)C(C)(C)C'] * 2})
    submission_df = pd.DataFrame({'ID': test_df['ID'], 'Inhibition': 0})

y_train = train_df['Inhibition'].values
print("데이터 로드 완료.\n")

In [ ]:
def normalized_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    range_y = np.max(y_true) - np.min(y_true)
    return rmse / range_y if range_y != 0 else rmse

def competition_score(y_true, y_pred):
    A = normalized_rmse(y_true, y_pred)
    B, _ = pearsonr(y_true, y_pred) if len(np.unique(y_pred)) >= 2 else (0.0, 0.0)
    score = 0.5 * (1 - min(A, 1)) + 0.5 * B
    return score, A, B

def plot_tsne(features, labels, title):
    print(f"t-SNE 시각화 진행 중: '{title}'...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
    tsne_results = tsne.fit_transform(features)
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=labels, cmap='viridis', alpha=0.7)
    plt.title(title, fontsize=16)
    plt.xlabel("t-SNE Component 1"); plt.ylabel("t-SNE Component 2")
    cbar = plt.colorbar(scatter); cbar.set_label('Inhibition (%)')
    plt.grid(True, linestyle='--', alpha=0.6); plt.show()

def plot_predictions_vs_actual(y_true, y_pred, title):
    score, _, _ = competition_score(y_true, y_pred)
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, edgecolor='k', s=50)
    min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction (y=x)')
    plt.title(f'{title}\nCompetition Score: {score:.4f}', fontsize=16)
    plt.xlabel('실제 값 (Actual Inhibition %)', fontsize=12)
    plt.ylabel('예측 값 (Predicted Inhibition %)', fontsize=12)
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6); plt.show()


In [ ]:
print("===== 2. RDKit 기반 특징 공학 시작 =====")
def calculate_rdkit_features(df):
    rdkit_desc, morgan_fp = [], []
    for smiles in tqdm(df['Canonical_Smiles'], desc="RDKit Features"):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rdkit_desc.append([np.nan] * 18)
            morgan_fp.append(np.zeros((1024,), dtype=int))
            continue
        # Descriptors
        desc = [ Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.NumHAcceptors(mol), Descriptors.NumHDonors(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol), Descriptors.NumHeteroatoms(mol), Descriptors.FractionCSP3(mol), Descriptors.NumAliphaticRings(mol), Lipinski.NumAromaticHeterocycles(mol), Lipinski.NumSaturatedHeterocycles(mol), Lipinski.NumAliphaticHeterocycles(mol), Descriptors.HeavyAtomCount(mol), Descriptors.RingCount(mol), Descriptors.NOCount(mol), Descriptors.NHOHCount(mol), Descriptors.NumRadicalElectrons(mol) ]
        rdkit_desc.append(desc)
        # Fingerprints
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        morgan_fp.append(np.array(fp))
    return pd.DataFrame(rdkit_desc, columns=descriptor_columns), np.array(morgan_fp)

descriptor_columns = [ 'MolWt', 'MolLogP', 'NumHAcceptors', 'NumHDonors', 'TPSA', 'NumRotatableBonds', 'NumAromaticRings', 'NumHeteroatoms', 'FractionCSP3', 'NumAliphaticRings', 'NumAromaticHeterocycles', 'NumSaturatedHeterocycles', 'NumAliphaticHeterocycles', 'HeavyAtomCount', 'RingCount', 'NOCount', 'NHOHCount', 'NumRadicalElectrons' ]
train_desc, train_fp = calculate_rdkit_features(train_df)
test_desc, test_fp = calculate_rdkit_features(test_df)

for col in descriptor_columns:
    mean_val = train_desc[col].mean()
    train_desc[col].fillna(mean_val, inplace=True); test_desc[col].fillna(mean_val, inplace=True)

scaler = StandardScaler()
train_desc_scaled = scaler.fit_transform(train_desc)
test_desc_scaled = scaler.transform(test_desc)
print("RDKit 기반 특징 공학 완료.\n")

In [ ]:
print("===== 3. ChemBERTa 임베딩 추출 및 압축 시작 =====")

# --- 이 부분을 수정합니다 ---
# 기존 모델명
# model_name = "DeepChem/ChemBERTa-77M-MTR"

# 1순위 추천 모델 (ChEMBL 기반)
model_name = "jonghyunlee/ChemBERT_ChEMBL_pretrained"

# 2순위 추천 모델 (대안)
# model_name = "seyonec/ChemBERTa-zinc-base-v1"
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(model_name)
chemberta_model = AutoModel.from_pretrained(model_name).to(device)

def get_chemberta_embeddings(smiles_list, model, tokenizer, batch_size=64):
    model.eval()
    all_embeddings = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc="ChemBERTa Embeddings"):
        inputs = tokenizer(smiles_list[i:i+batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

train_chemberta_embeddings = get_chemberta_embeddings(train_df['Canonical_Smiles'].tolist(), chemberta_model, tokenizer)
test_chemberta_embeddings = get_chemberta_embeddings(test_df['Canonical_Smiles'].tolist(), chemberta_model, tokenizer)
print("ChemBERTa 임베딩 추출 완료.")

# --- 5.1. 압축 전 t-SNE 시각화 ---
plot_tsne(train_chemberta_embeddings, y_train, 'ChemBERTa 원본 임베딩 t-SNE')

In [ ]:
embedding_dim = train_chemberta_embeddings.shape[1]
latent_dim = 16

print(f"\nAutoencoder 입력 차원 (ChemBERTa 임베딩 차원): {embedding_dim}")
print(f"Autoencoder 압축 후 목표 차원: {latent_dim}")

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), 
                                    nn.Linear(256, 128), nn.ReLU(),
                                    nn.Linear(128, 64), nn.ReLU(),
                                    nn.Linear(64, 32), nn.ReLU(),
                                    nn.Linear(32, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(),
                                    nn.Linear(32, 64), nn.ReLU(),
                                    nn.Linear(64, 128), nn.ReLU(),
                                    nn.Linear(128, 256), nn.ReLU(),
                                    nn.Linear(256, input_dim))
    def forward(self, x): return self.decoder(self.encoder(x))

ae_model = Autoencoder(embedding_dim, latent_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-5)
train_loader = DataLoader(TensorDataset(torch.tensor(train_chemberta_embeddings, dtype=torch.float32)), batch_size=64, shuffle=True)

print("Autoencoder 학습 중...")
for epoch in tqdm(range(50), desc="AE Training"):
    for data in train_loader:
        inputs = data[0].to(device)
        outputs = ae_model(inputs)
        loss = criterion(outputs, inputs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
print("Autoencoder 학습 완료.")

def get_latent_features(model, embeddings):
    model.eval()
    with torch.no_grad():
        return model.encoder(torch.tensor(embeddings, dtype=torch.float32).to(device)).cpu().numpy()

train_latent_features = get_latent_features(ae_model, train_chemberta_embeddings)
test_latent_features = get_latent_features(ae_model, test_chemberta_embeddings)
print("임베딩 압축 완료.\n")

# --- 5.3. 압축 후 t-SNE 시각화 ---
plot_tsne(train_latent_features, y_train, 'AE 압축 특징 t-SNE')

In [ ]:
print("===== 4. 최종 특징 결합 시작 =====")
X_train = np.hstack([train_desc_scaled, train_fp, train_latent_features])
X_test = np.hstack([test_desc_scaled, test_fp, test_latent_features])


X_train = np.hstack([train_desc_scaled, train_latent_features])
X_test = np.hstack([test_desc_scaled, test_latent_features])

print(f"Final Train Features Shape: {X_train.shape}")
print(f"Final Test Features Shape: {X_test.shape}")
print("최종 특징 결합 완료.\n")

In [ ]:
import optuna
print("===== 5. 전체 모델 하이퍼파라미터 튜닝 (Optuna) 시작 =====")
n_splits_tuning = 5 # 튜닝 시간을 줄이기 위해 Fold 수를 줄일 수 있습니다. (예: 3)
kf_tuning = KFold(n_splits=n_splits_tuning, shuffle=True, random_state=42)

def run_optimization(model_name, objective_func, n_trials=50):
    """모델별 Optuna 최적화를 실행하는 헬퍼 함수"""
    print(f"\n--- {model_name} 하이퍼파라미터 튜닝 시작 ---")
    study = optuna.create_study(direction='maximize', study_name=f'{model_name}_optimization')
    study.optimize(objective_func, n_trials=n_trials)
    
    print(f"--- {model_name} 튜닝 완료 ---")
    print(f"최고 점수 (CV Score): {study.best_value:.4f}")
    print(f"최적 하이퍼파라미터: {study.best_params}")
    return study.best_params

# --- 7.1. RandomForest Objective 함수 ---
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 42, 'n_jobs': -1
    }
    model = RandomForestRegressor(**params)
    oof_preds_trial = np.zeros(X_train.shape[0])
    for train_idx, val_idx in kf_tuning.split(X_train):
        model.fit(X_train[train_idx], y_train[train_idx])
        oof_preds_trial[val_idx] = model.predict(X_train[val_idx])
    score, _, _ = competition_score(y_train, oof_preds_trial)
    return score

# --- 7.2. LightGBM Objective 함수 ---
def objective_lgbm(trial):
    params = {
        'objective': 'regression_l1', 'metric': 'rmse', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
    }
    model = lgb.LGBMRegressor(**params)
    oof_preds_trial = np.zeros(X_train.shape[0])
    for train_idx, val_idx in kf_tuning.split(X_train):
        model.fit(X_train[train_idx], y_train[train_idx], eval_set=[(X_train[val_idx], y_train[val_idx])], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_preds_trial[val_idx] = model.predict(X_train[val_idx])
    score, _, _ = competition_score(y_train, oof_preds_trial)
    return score

# --- 7.3. XGBoost Objective 함수 ---
def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'tree_method': 'gpu_hist' if 'cuda' in device.type else 'hist',
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }
    model = xgb.XGBRegressor(**params)
    oof_preds_trial = np.zeros(X_train.shape[0])
    for train_idx, val_idx in kf_tuning.split(X_train):
        # early_stopping_rounds를 fit의 인자로 직접 전달
        model.fit(X_train[train_idx], y_train[train_idx],
                  eval_set=[(X_train[val_idx], y_train[val_idx])],
                  verbose=False)
        oof_preds_trial[val_idx] = model.predict(X_train[val_idx])
    score, _, _ = competition_score(y_train, oof_preds_trial)
    return score
    
# --- 7.4. GradientBoosting Objective 함수 ---
def objective_gb(trial):
    params = {
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    model = GradientBoostingRegressor(**params)
    oof_preds_trial = np.zeros(X_train.shape[0])
    for train_idx, val_idx in kf_tuning.split(X_train):
        model.fit(X_train[train_idx], y_train[train_idx])
        oof_preds_trial[val_idx] = model.predict(X_train[val_idx])
    score, _, _ = competition_score(y_train, oof_preds_trial)
    return score

# --- 7.5. 각 모델에 대한 최적화 실행 ---
# n_trials를 조절하여 탐색 시간과 성능 사이의 균형을 맞출 수 있습니다. (예: 20~50)
best_params_rf = run_optimization("RandomForest", objective_rf, n_trials=30)
best_params_lgbm = run_optimization("LightGBM", objective_lgbm, n_trials=30)
best_params_xgb = run_optimization("XGBoost", objective_xgb, n_trials=30)
best_params_gb = run_optimization("GradientBoosting", objective_gb, n_trials=30)

In [ ]:
print("\n===== 6. 최적화된 모델로 최종 학습 및 비교 시작 =====")
n_splits_final = 5 # 최종 학습에는 더 안정적인 5-Fold 사용
kf_final = KFold(n_splits=n_splits_final, shuffle=True, random_state=42)

# 최적화된 파라미터로 모델 딕셔너리 재구성
models = {
    "Optimized_RF": RandomForestRegressor(**best_params_rf, random_state=42, n_jobs=-1),
    "Optimized_LGBM": lgb.LGBMRegressor(**best_params_lgbm, random_state=42, verbosity=-1, n_jobs=-1),
    "Optimized_XGB": xgb.XGBRegressor(**best_params_xgb, random_state=42, n_jobs=-1, tree_method='gpu_hist' if 'cuda' in device.type else 'hist'),
    "Optimized_GB": GradientBoostingRegressor(**best_params_gb, random_state=42)
}

oof_preds, test_preds, model_scores = {}, {}, {}

for name, model in models.items():
    print(f"\n--- {name} 최종 학습 ---")
    current_oof = np.zeros(len(train_df))
    current_test = np.zeros(len(test_df))
    
    for fold, (train_idx, val_idx) in enumerate(kf_final.split(X_train)):
        print(f"  Fold {fold+1}/{n_splits_final}", end=" ")
        X_train_fold, y_train_fold = X_train[train_idx], y_train[train_idx]
        X_val_fold, y_val_fold = X_train[val_idx], y_train[val_idx]
        
        if "LGBM" in name:
            model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)],
                    callbacks=[lgb.early_stopping(50, verbose=False)])
        elif "XGB" in name:
            # XGBoost의 fit 메소드에 early_stopping_rounds를 직접 사용
            model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)],
                    verbose=False)
        else:
            model.fit(X_train_fold, y_train_fold)
            
        current_oof[val_idx] = model.predict(X_val_fold)
        current_test += model.predict(X_test) / n_splits_final
        print("Done.")

    oof_preds[name] = np.clip(current_oof, 0, 100)
    test_preds[name] = np.clip(current_test, 0, 100)
    score, _, _ = competition_score(y_train, oof_preds[name])
    model_scores[name] = score
    print(f"  {name} 최종 OOF Competition Score: {score:.4f}")


In [ ]:
print("\n===== 7. 최종 모델 선택 및 제출 =====")

# --- 방법 1: 최고 성능 단일 모델 선택 ---
best_model_name = max(model_scores, key=model_scores.get)
print("\n--- 모델별 최종 OOF 점수 (단일 모델) ---")
for name, score in sorted(model_scores.items(), key=lambda item: item[1], reverse=True):
    print(f"{name}: {score:.4f}")

print(f"\n최고 성능 단일 모델: {best_model_name} (Score: {model_scores[best_model_name]:.4f})")
plot_predictions_vs_actual(y_train, oof_preds[best_model_name], f'OOF Predictions ({best_model_name})')
final_test_preds = test_preds[best_model_name]

In [ ]:
# --- 방법 2: 가중 평균 앙상블 (성능 향상을 위해 시도해볼 만한 옵션) ---
print("\n--- 가중 평균 앙상블 수행 ---")
# 점수를 가중치로 사용하기 위해 정규화
total_score = sum(model_scores.values())
weights = {name: score / total_score for name, score in model_scores.items()}

ensemble_test_preds = np.zeros(len(test_df))
ensemble_oof_preds = np.zeros(len(train_df))

for name, weight in weights.items():
    print(f"  {name} 가중치: {weight:.4f}")
    ensemble_oof_preds += oof_preds[name] * weight
    ensemble_test_preds += test_preds[name] * weight

ensemble_score, _, _ = competition_score(y_train, np.clip(ensemble_oof_preds, 0, 100))
print(f"\n앙상블 OOF Competition Score: {ensemble_score:.4f}")
plot_predictions_vs_actual(y_train, np.clip(ensemble_oof_preds, 0, 100), 'OOF Predictions (Ensemble)')
final_test_preds = np.clip(ensemble_test_preds, 0, 100) # 최종 제출 예측을 앙상블 결과로 교체

In [ ]:
submission_df['Inhibition'] = final_test_preds
submission_df.to_csv('submission_0712.csv', index=False)
print("\nsubmission.csv 파일이 성공적으로 생성되었습니다.")
print("===== 모든 프로세스 완료 =====")